# Broadcast camera to a 2D pitch

One tactical camera goes in. Tracked players, the goalkeeper, the referee, and the ball come out on a measured pitch.

The video for this notebook is YOLOv26x on `08fd33_0.mp4` (30 s, 25 fps):

`outputs/08fd33_yolov26x_h264.mp4`

Play that file in a video player. Cursor on WSL cannot embed the mp4 inside the notebook. The cells below are the code that builds it. `WRITE_VIDEO` stays `False` so running the notebook does not start another 15-minute render.

Kernel: `soccernet_atlas/.venv`, Python 3.11. Run the cells from the top.


## What the frame shows

![YOLOv26x: broadcast view and 2D pitch](figures/y26x_f450.jpg)

Left side: boxes and track ids. Right side: a FIFA pitch, 105 m by 68 m.

* White and red dots are the two teams, split from jersey colour.
* Green is the goalkeeper class from the detector.
* Yellow crosses are referees.
* The yellow disc with a white ring is the ball. The line behind it is the path the tracker accepted.


In [ ]:
from pathlib import Path
import sys
from collections import defaultdict

import cv2
import numpy as np
import torch
from matplotlib import pyplot as plt
from tqdm.auto import tqdm

ROOT = Path("/home/amani/phd/soccernet_atlas")
if not (ROOT / "atlas").is_dir():
    ROOT = Path.cwd()
sys.path.insert(0, str(ROOT))

from atlas.ball import BallTracker, draw_ball_trail
from atlas.minimap import apply_H, draw_empty_pitch, stamp_objects
from atlas.pitch import SoccerNetPitchSeg, estimate_H, get_pitch_keypoint_calibrator
from atlas.pitch_keypoints import PitchPosSmoother
from atlas.teams import assign_teams, jersey_lab, reset_team_model
from atlas.track_ids import TrackIdStabilizer

# Colab YOLOv26x detector. Same four classes as the YOLOv8x run.
WEIGHTS = ROOT / "yolov26x" / "best (3).pt"
PITCH_WEIGHTS = ROOT / "weights" / "football-pitch-detection.pt"
VIDEO = ROOT / "08fd33_0.mp4"
TRACKER = ROOT / "cfg" / "botsort_football.yaml"
OUT = ROOT / "outputs" / "08fd33_yolov26x_notebook.mp4"
DEMO = ROOT / "outputs" / "08fd33_yolov26x_h264.mp4"
IMGSZ = 1280
CONF = 0.25
MAX_FRAMES = 750
WRITE_VIDEO = False  # True rebuilds OUT. About 15 min on a 4 GB GPU.

TEAM_COLORS = {0: (255, 255, 255), 1: (0, 0, 255), None: (0, 215, 255)}  # BGR
GK_COLOR = (0, 200, 0)
BALL_COLOR = (0, 220, 255)
BALL_NAMES = {"ball", "sports ball"}
PERSON_NAMES = {"person", "player", "referee", "goalkeeper", "goalie"}
GK_NAMES = {"goalkeeper", "goalie"}

print("demo:", DEMO)
print("exists" if DEMO.exists() else "MISSING")
still = cv2.imread(str(ROOT / "notebooks" / "figures" / "y26x_f450.jpg"))
plt.figure(figsize=(14, 4))
plt.imshow(cv2.cvtColor(still, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.tight_layout()
plt.show()


## What we built

Each frame does five things.

1. **Detect.** A YOLO detector finds `ball`, `goalkeeper`, `player`, and `referee`.
2. **Track.** BoT-SORT (`cfg/botsort_football.yaml`) keeps an id when a player disappears for a short time. The buffer is 90 frames, which is what stops the goalkeeper changing id every time the box flickers. `TrackIdStabilizer` can also reuse a lost id if the new box lands within 7 m of it on the pitch.
3. **Calibrate.** A second network finds 32 pitch landmarks. Each landmark has a known FIFA coordinate. MAGSAC fits a homography from image pixels to metres. It runs every frame because the broadcast camera moves. SoccerNet's line segmenter is only a fallback if the keypoint fit fails.
4. **Teams.** Outfield players are clustered by jersey colour in Lab space. The goalkeeper and the referee stay on the class the detector gave them.
5. **Ball.** The ball is not taken from the tracker. BoT-SORT drops tiny low-confidence boxes, so the same network is run a second time at confidence 0.05, class `ball` only. A box is drawn only when it is small, sitting on grass, inside the pitch, and near the previous ball. Otherwise the trail breaks instead of jumping across the map.

The pitch is 105 × 68 m. An earlier version scaled a 120 × 70 diagram and stretched players toward the touchlines. That scaling is gone.


## Detectors we trained and compared

All of the football detectors were trained on the Roboflow set `football-players-detection` (the version used in Colab is version 12), four classes, input size 1280.

| Weights | What it is | Parameters | Role |
|---|---|---:|---|
| `runs/detect/train/weights/best.pt` | YOLO26n, trained locally | 2.5M | Fast baseline. A 30 s clip renders in a few minutes. |
| `runs/detect/train_gk/weights/best.pt` | YOLO26n fine-tuned on a goalkeeper-balanced copy of the set | 2.5M | 40 epochs. Best validation epoch was 20 (mAP50 0.901). Not the model in the demo video. |
| `yolov8/best.pt` | YOLOv8x, trained on Colab | 68M | Clearer ball boxes than the nano model. Slow on a 4 GB GPU. |
| `yolov26x/best (3).pt` | YOLOv26x, trained on Colab | 59M | The video above. Same four classes, `imgsz=1280`. |

On `08fd33_0.mp4`, every 15th frame, ball class only, confidence 0.05, before the grass and continuity gates:

| Model | Frames with a ball box | Mean confidence |
|---|---:|---:|
| YOLO26n | 37 / 50 | 0.36 |
| YOLOv8x | 39 / 50 | 0.52 |

Raising the input size above 1280 did not help the nano model on `data/sample.mp4` (20/30 frames at 1280, 19/30 at 1536, 18/30 at 1920). The demo stays at 1280.

Two earlier files in `yolov26x/` are not this detector. `best (1).pt` and `best (2).pt` are old YOLOv8 segmentation checkpoints with a single class named `pp`, trained at size 640. They came from `runs/segment` on Drive. They do not detect players or the ball. The football YOLOv26x run writes `runs/detect/.../weights/best.pt`. On Colab that directory lives under `/content` and is deleted when the runtime stops, so the training command now sets `project` on Google Drive, or downloads `best.pt` with `files.download` at the end of the cell.


## 1. Load YOLOv26x

`ultralytics.YOLO` rebuilds the network from the checkpoint and uses the GPU when CUDA is available. The four class ids are fixed for the rest of the pipeline: `0` ball, `1` goalkeeper, `2` player, `3` referee.


In [ ]:
from ultralytics import YOLO

model = YOLO(str(WEIGHTS))
n_params = sum(p.numel() for p in model.model.parameters())
print("task   ", model.task)
print("classes", model.names)
print(f"params  {n_params/1e6:.1f} M")
print("cuda   ", torch.cuda.is_available())


## 2. Load the pitch-keypoint model

`weights/football-pitch-detection.pt` is a separate pose model with 32 landmarks (corners, penalty spots, centre circle, halfway line). `get_pitch_keypoint_calibrator` loads it. `estimate_H` keeps centre-circle and halfway points first, ignores a corner unless its confidence is very high, and fits `H` with MAGSAC. A bad corner is what was stretching the minimap.

`SoccerNetPitchSeg` loads the public DeepLab line model. It is called only when the keypoint fit returns nothing.


In [ ]:
kp = get_pitch_keypoint_calibrator(reset=True)
print("pitch weights", PITCH_WEIGHTS.name, PITCH_WEIGHTS.exists(), "task", kp.model.task)

try:
    seg = SoccerNetPitchSeg()
    print("SoccerNet line model loaded (fallback only)")
except Exception as e:
    seg = None
    print("SoccerNet fallback not loaded:", type(e).__name__, e)


## 3. Open the clip

`BallTracker` holds the ball state: confidence 0.05, a trail of 500 points, 12 frames of hold when the detector misses, and a 4.5 m jump limit so a false box cannot teleport across the pitch. `PitchPosSmoother` damps player dots without lagging a camera pan. `TrackIdStabilizer` reclaims a lost id inside 7 m and 75 frames.


In [ ]:
reset_team_model()
cap = cv2.VideoCapture(str(VIDEO))
fps = cap.get(cv2.CAP_PROP_FPS) or 25
W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
n = min(int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) or MAX_FRAMES, MAX_FRAMES)
print(f"{VIDEO.name}: {n} frames, {W}x{H} @ {fps:.0f} fps")

pitch_blank, xy = draw_empty_pitch(h=H, w=int(H * 105 / 68))
map_w = pitch_blank.shape[1]
pos_smooth = PitchPosSmoother(alpha=0.75, max_jump_m=10.0)
id_stable = TrackIdStabilizer(max_dist_m=7.0, max_gap_frames=75)
ball_tr = BallTracker(conf_thr=0.05, trail_len=500, max_coast_frames=12, max_jump_m=4.5)
lab_hist = defaultdict(list)
last_H = None
names = model.names

writer = None
if WRITE_VIDEO:
    OUT.parent.mkdir(parents=True, exist_ok=True)
    writer = cv2.VideoWriter(str(OUT), cv2.VideoWriter_fourcc(*"mp4v"), fps, (W + map_w, H))
    model.predictor = None
    print("writing", OUT)
else:
    print("WRITE_VIDEO is False. The loop in the next cell is the writer; it will not run.")
    print("Play", DEMO)


## 4. The loop that writes the video

This is the body of `run.py`, pointed at the YOLOv26x weights.

For every frame: track people at confidence 0.25, detect the ball on its own, estimate `H`, project feet and the ball into metres, draw the minimap, and append one side-by-side frame. The foot point is the bottom centre of the box, lifted by 2% so shoes and shadows do not sit on the line.


In [ ]:
if WRITE_VIDEO:
    for _ in tqdm(range(n)):
        ok, frame = cap.read()
        if not ok:
            break

        result = model.track(
            frame, persist=True, tracker=str(TRACKER),
            conf=CONF, imgsz=IMGSZ, verbose=False,
        )[0]
        vis = frame.copy()
        objects = []
        ball_det_img = ball_tr.detect_image(model, frame, imgsz=IMGSZ)

        if result.boxes is not None and result.boxes.id is not None:
            for box, c, tid in zip(
                result.boxes.xyxy.cpu().numpy(),
                result.boxes.cls.cpu().numpy().astype(int),
                result.boxes.id.cpu().numpy().astype(int),
            ):
                cname = str(names.get(int(c), "")).lower()
                if cname in BALL_NAMES or cname not in PERSON_NAMES:
                    continue
                if cname == "referee":
                    kind = "referee"
                elif cname in GK_NAMES:
                    kind = "goalkeeper"
                else:
                    kind = "player"
                x1, y1, x2, y2 = box
                foot = np.array([(x1 + x2) / 2.0, y2 - 0.02 * max(y2 - y1, 1.0)], np.float32)
                if kind == "player":
                    feat = jersey_lab(frame, box)
                    if feat is not None:
                        lab_hist[int(tid)].append(feat)
                objects.append(dict(kind=kind, tid=int(tid), box=box, foot=foot, team=None))

        means = {t: np.mean(v, axis=0) for t, v in lab_hist.items() if len(v) >= 3}
        teams = assign_teams(means) if means else {}
        for o in objects:
            if o["kind"] == "player":
                o["team"] = teams.get(o["tid"])

        Hmat = estimate_H(frame, seg, use_keypoints=True, prefer_soccernet=False)
        if Hmat is not None:
            last_H = Hmat

        for o in objects:
            if last_H is None:
                o["x_m"] = o["y_m"] = None
                continue
            p = apply_H(last_H, o["foot"].reshape(1, 2))[0]
            o["x_m"] = float(np.clip(p[0], -3.0, 108.0))
            o["y_m"] = float(np.clip(p[1], -3.0, 71.0))
        id_stable.remap(objects)

        live = set()
        for o in objects:
            if o["x_m"] is not None:
                sm = pos_smooth.update(int(o["tid"]), np.array([o["x_m"], o["y_m"]], np.float32))
                o["x_m"], o["y_m"] = float(sm[0]), float(sm[1])
                live.add(int(o["tid"]))
            x1, y1, x2, y2 = o["box"]
            if o["kind"] == "goalkeeper":
                color, label = GK_COLOR, f"GK {o['tid']}"
            elif o["kind"] == "referee":
                color, label = TEAM_COLORS[None], f"ref {o['tid']}"
            else:
                color, label = TEAM_COLORS.get(o["team"], (200, 200, 200)), f"ID {o['tid']}"
            cv2.rectangle(vis, (int(x1), int(y1)), (int(x2), int(y2)), (20, 20, 20), 3)
            cv2.rectangle(vis, (int(x1), int(y1)), (int(x2), int(y2)), color, 2)
            cv2.putText(vis, label, (int(x1), max(15, int(y1) - 6)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1, cv2.LINE_AA)
        pos_smooth.prune(live)

        ball_xy_m = None
        if ball_det_img is not None and last_H is not None:
            img_xy, bconf = ball_det_img
            proj = apply_H(last_H, img_xy.reshape(1, 2).astype(np.float32))[0]
            if -2.0 <= float(proj[0]) <= 107.0 and -2.0 <= float(proj[1]) <= 70.0:
                ball_xy_m = proj
                ball_tr.last_img_xy = img_xy
                ix, iy = int(img_xy[0]), int(img_xy[1])
                cv2.circle(vis, (ix, iy), 10, BALL_COLOR, 2)
                cv2.putText(vis, f"ball {bconf:.2f}", (ix + 12, iy - 8),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, BALL_COLOR, 1, cv2.LINE_AA)
        ball_pos = ball_tr.update(None if ball_xy_m is None else np.asarray(ball_xy_m, np.float64))

        mini = stamp_objects(pitch_blank, xy, objects)
        draw_ball_trail(mini, xy, list(ball_tr.trail), color=BALL_COLOR)
        if ball_pos is not None:
            p = xy(float(np.clip(ball_pos[0], 0, 105)), float(np.clip(ball_pos[1], 0, 68)))
            cv2.circle(mini, p, 12, (255, 255, 255), 2, cv2.LINE_AA)
            cv2.circle(mini, p, 8, BALL_COLOR, -1)
        if mini.shape[0] != H:
            mini = cv2.resize(mini, (map_w, H))
        cv2.putText(vis, "YOLOv26x + BoT-SORT + ball trail + keypoint H", (12, 28),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2, cv2.LINE_AA)
        writer.write(np.hstack([vis, mini]))

    cap.release()
    writer.release()
    print("wrote", OUT)
else:
    cap.release()
    print("render skipped")
    print("open", DEMO)


## Why the ball is a second pass

The first videos drew a scribble, or no ball at all. The detector was returning a ball box on many frames, but the highest box was often a boot, a shirt, or a logo. Connecting those boxes made a line across the whole right half of the pitch.

`BallTracker.detect_image` now keeps a box only if:

* it is a few dozen pixels across, not a person-sized box
* the patch around it is grass
* after projection, it lies inside the 105 × 68 pitch, with a 2 m margin
* it stays near the previous ball in the image and moves at most about 4.5 m per frame on the pitch

A miss holds the last point for 12 frames and does not extend the line. That is why the trail on the minimap breaks instead of cutting across the centre circle.

The same gates are in `atlas/ball.py` and are what `run.py` calls.


## Reproduce

```bash
cd /home/amani/phd/soccernet_atlas
source .venv/bin/activate
python run.py \
  --video 08fd33_0.mp4 \
  --weights "yolov26x/best (3).pt" \
  --out outputs/08fd33_yolov26x.mp4 \
  --imgsz 1280 \
  --max-frames 750
```

`run.py` writes MPEG-4. The file to play is the H.264 copy, `outputs/08fd33_yolov26x_h264.mp4`. The same command with `yolov8/best.pt` writes the YOLOv8x clip (`outputs/08fd33_yolov8x_h264.mp4`). The nano clip is `outputs/08fd33_yolo26n_h264.mp4`.

Colab training for the detector, with the checkpoint kept after the runtime stops:

```python
HOME = "/content"
!mkdir -p {HOME}/datasets
# download football-players-detection version 12 in yolo26 format, then:
!yolo task=detect mode=train model=yolo26x.pt data="{dataset.location}/data.yaml" \
  batch=6 epochs=50 imgsz=1280 plots=True \
  project="/content/runs/yolo26x" name=football exist_ok=True
from google.colab import files
files.download("/content/runs/yolo26x/football/weights/best.pt")
```

A finished detector has task `detect` and classes `ball`, `goalkeeper`, `player`, `referee`. A file whose only class is `pp` is a segmentation run and cannot drive this notebook.


## Limits

* One camera. No multi-view fusion and no event detection.
* The ball is still missed for stretches. The trail stops through those gaps.
* Players near the edge of the frame can sit a few metres off, because the homography is extrapolated from the landmarks that are visible.
* YOLOv26x and YOLOv8x each take about 15 minutes for this 30-second clip on a 4 GB laptop GPU. The nano model is the fast one and is weaker on the ball.
* Team colour is two clusters for this match, white and red. It is not a general kit classifier.
